#Arquivo criado para hospedar os códigos e respostas das perguntas 1 a 10 do arquivo "perguntas_sql.md" em Python.

**#Primeiro passo**: importação das bases de dados do GCP.

In [ ]:
import basedosdados as bd
import pandas as pd

PROJECT_ID = "desafio-empm-ulisses-gomes" 

print("Baixando dados do GCP")

# Baixando as tabelas 'dados_mestres_bairro' e 'turismo_fluxo_visitantes.rede_hoteleira_ocupacao_eventos'.
df_bairros = bd.read_sql("SELECT * FROM `datario.dados_mestres.bairro`", billing_project_id=PROJECT_ID)
df_eventos = bd.read_sql("SELECT * FROM `datario.turismo_fluxo_visitantes.rede_hoteleira_ocupacao_eventos`", billing_project_id=PROJECT_ID)

#Baixando a tabela 'adm_central_atendimento_1746.chamado' com o filtros para não exceder a cota de consultas.
#Para as perguntas de 1 a 5, baixamos apenas os chamados do dia 01/04/2023
query_p1 = "SELECT * FROM `datario.adm_central_atendimento_1746.chamado` WHERE DATE(data_inicio) = '2023-04-01'"
df_chamados_p1 = bd.read_sql(query_p1, billing_project_id=PROJECT_ID)
#Para as perguntas 6 a 10, baixamos apenas os chamados de Perturbação do Sossego entre 2022 e 2024.
query_p2 = """
SELECT * FROM `datario.adm_central_atendimento_1746.chamado` 
WHERE id_subtipo = '5071' AND DATE(data_inicio) BETWEEN '2022-01-01' AND '2024-12-31'
"""
df_chamados_p2 = bd.read_sql(query_p2, billing_project_id=PROJECT_ID)

#Transformando as colunas de datas para o formato 'datetime.date' do pandas.
df_chamados_p2['data_inicio'] = pd.to_datetime(df_chamados_p2['data_inicio']).dt.date
df_eventos['data_inicial'] = pd.to_datetime(df_eventos['data_inicial']).dt.date
df_eventos['data_final'] = pd.to_datetime(df_eventos['data_final']).dt.date

print("Bases carregadas")

Baixando dados do GCP
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Bases carregadas


**#PERGUNTA 1**- Quantos chamados foram registrados no dia 01/04/2023?

In [ ]:
#Objetivo do código, aplicar 'len' para contar o número de chamados registrados no dia 01/04/2023.
q1_resposta = len(df_chamados_p1)
print(f"Resposta: foram registrados {q1_resposta} chamados em 01/04/2023:")

Resposta: Total de chamados no dia 01/04/2023: 2067


**#PERGUNTA 2**- Qual foi o bairro com o maior número de chamados registrados no dia 01/04/2023?

In [6]:
# Objetivo da código: aplicar 'valuecounts e 'idxmax' para contar os chamados e retornar apenas o chamado mais registrado
q2_resposta = df_chamados_p1['tipo'].value_counts().idxmax()
print(f"Resposta: o chamado mais registrado foi o de '{q2_resposta}'")

Resposta: o chamado mais registrado foi o de 'Estacionamento irregular'


#Mesclagem 'merge' das tabelas 'dados_mestres_bairro' e 'adm_central_atendimento_1746.chamado'

In [8]:
df_merged_p1 = df_chamados_p1.merge(df_bairros, on='id_bairro', how='left')
print("Mesclagem executada")

Mesclagem executada


**#PERGUNTA 3**- Quais os nomes dos 3 bairros que mais tiveram chamados abertos nesse dia?

In [18]:
#objetivo do código: aplicar 'valuecounts' para contar os chamados por bairro e 'head(3)' para retornar apenas os 3 maiores.
q3_resposta = df_merged_p1['nome'].value_counts().head(3) #Retorna apenas os 3 bairros geo-referenciados nas reclamações com mais chamados registrados.
print("\n" + "-" * 50)
print("RESULTADO: TOP 3 BAIRROS")
print("-" * 50)
for posicao, (bairro, quantidade) in enumerate(q3_resposta.items(), start=1): #Itera por meio do 'enumerate' sobre os bairros e suas respectivas quantidades, formatando a saída para exibir a posição, nome do bairro e quantidade de chamados.
    print(f" {posicao}º Lugar | {bairro.ljust(25)} | {quantidade} chamados")

print("-" * 50 + "\n")


--------------------------------------------------
RESULTADO: TOP 3 BAIRROS
--------------------------------------------------
 1º Lugar | Campo Grande              | 125 chamados
 2º Lugar | Tijuca                    | 100 chamados
 3º Lugar | Barra da Tijuca           | 62 chamados
--------------------------------------------------



**Observação** Como descobrimos na nossa exploração inicial em SQL (Pergunta 3 e 5), existem chamados no sistema 1746 que não possuem um 'id_bairro' associado. Geralmente, são serviços atrelados a entidades móveis (como reclamações de linhas de Ônibus ou BRT) ou serviços digitais, que não possuem geolocalização fixa. 

Como a função 'value_counts()' tem como parâmetro o 'dropna=True', que desconsidera os valores nulos, optou-se aqui pelo uso dessa função e assim não evidenciar as reclamações não geolocalizadas, como foi evidenciado na consulta SQL.

**PERGUNTA 4**- Qual o nome da subprefeitura com mais chamados abertos nesse dia?

In [ ]:
#Objetivo do código: aplicar 'valuecounts' para contar os chamados por subprefeitura e 'idxmax' para retornar apenas a subprefeitura com mais chamados registrados.
q4_resposta = df_merged_p1['subprefeitura'].value_counts().idxmax()
print(f"\nResposta: A subprefeitura com mais chamados foi a '{q4_resposta}'")


Resposta: A subprefeitura com mais chamados foi a 'Subprefeitura da Zona Norte'


**PERGUNTA 5:** - Existe algum chamado aberto nesse dia que não foi associado a um bairro ou subprefeitura na tabela de bairros? Se sim, por que isso acontece?

In [26]:
chamados_sem_bairro = df_merged_p1[df_merged_p1['id_bairro'].isnull()]
print(f"\nResultado: Sim, existem {len(chamados_sem_bairro)} chamados sem bairro associado.")
print("Motivação:Ao analisar a consulta, observa-se que esses chamados se referem a trasnportes como 'Veículos' e /ou 'BRT (corredor expresso de ônibus)' - de maneira similar a pergunta 3 - 'Fiscalização Eletrônica' e 'Ouvidoria - CLF'")


Resultado: Sim, existem 260 chamados sem bairro associado.
Motivação:Ao analisar a consulta, observa-se que esses chamados se referem a trasnportes como 'Veículos' e /ou 'BRT (corredor expresso de ônibus)' - de maneira similar a pergunta 3 - 'Fiscalização Eletrônica' e 'Ouvidoria - CLF'
